In [ ]:
!git clone https://github.com/vllm-project/speculators.git
%cd speculators

Cloning into 'speculators'...
remote: Enumerating objects: 15304, done.
remote: Counting objects: 100% (7075/7075), done.
remote: Compressing objects: 100% (2243/2243), done.
remote: Total 15304 (delta 6116), reused 4857 (delta 4818), pack-reused 8229 (from 2)
Receiving objects: 100% (15304/15304), 56.48 MiB | 17.94 MiB/s, done.
Resolving deltas: 100% (10248/10248), done.
/content/speculators


In [ ]:
# Speculators, data prep and training
!uv venv speculators
!uv pip install --python speculators/bin/python -e .

In [ ]:
# vLLM, for serving the target model
!uv venv vllm
!uv pip install --python vllm/bin/python "vllm>=0.27.1"

In [ ]:
%pip install -q datasets

In [ ]:
from datasets import load_dataset
from collections import Counter
import json

# ejemplos por idioma
cuanto = {
    "es": 25,
    "ca": 25,
    "eu": 25,
    "gl": 25,
}



In [ ]:
#Uso el dataset de SFT de ALIA, ya que incluye los idiomas que me interesan
dataset = load_dataset("BSC-LT/ALIA-2606-SFT",split="train",streaming=True).shuffle(seed=16, buffer_size=10000)

contar = Counter()

In [ ]:

#Info relevante para recorrer el dataset en Dataset Structure en HF

with open("alia_prompts_pilot.jsonl", "w", encoding="utf-8") as file:
    for fila in dataset:
        language = fila["lang"]

        #At this moment we are only interested in es, ca, eu and gl
        if language not in cuanto:
            continue

        #We don't want to skip our limit
        if contar[language] >= cuanto[language]:
            continue

        messages = fila["messages"]

        json.dump({"id": fila["id"],"messages": messages}, file, ensure_ascii=False)
        file.write("\n")
        contar[language] += 1

        #We finish when we got everything we need
        if all(contar[l] >= n for l, n in cuanto.items()):
            break

print("Conversations:", dict(contar))

Conversations: {'es': 25, 'ca': 25, 'eu': 25, 'gl': 25}


In [ ]:
!./vllm/bin/vllm --version

W1109 01:07:19.301000 5631 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1109 01:07:19.301000 5631 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1109 01:07:19.302000 5631 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
FlashInfer kernels are disabled: flashinfer-cubin is not installed and nvcc (CUDA_HOME, CUDA_PATH, PATH or /usr/local/cuda) or ninja (PATH) is missing. Set CUDA_HOME to a CUDA toolkit and put ninja on PATH, or run `vllm download-kernels`.
0.33.0


In [ ]:
#FlashInfer kernels are disabled fix
!./vllm/bin/vllm download-kernels

W1109 01:15:56.928000 8152 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1109 01:15:56.928000 8152 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1109 01:15:56.929000 8152 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
FlashInfer kernels are disabled: flashinfer-cubin is not installed and nvcc (CUDA_HOME, CUDA_PATH, PATH or /usr/local/cuda) or ninja (PATH) is missing. Set CUDA_HOME to a CUDA toolkit and put ninja on PATH, or run `vllm download-kernels`.
=== Cubin Wheel Install ===
FlashInfer version: 0.7.2.post1
Wheel index: https://flashinfer.ai/whl
Requirement: flashinfer-cubin==0.7.2.post1
Command: /usr/local/bin/uv pip install --python /content/speculators/vllm/bin/python --upgrade --no-deps --index-url https://flashinfer.ai/whl flashinfer-cubin==0.7.2.post1
Using Python 3.13.16 environment at: vllm
Resolved 1 package in 297ms
P

In [ ]:
#I use 8192 as max model len to follow the example of speculators
!./vllm/bin/vllm serve BSC-LT/salamandra-7b-instruct-2606 --max-model-len 8192 --port 8000

W1109 01:17:08.212000 8554 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1109 01:17:08.213000 8554 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1109 01:17:08.213000 8554 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
(APIServer pid=8554) INFO 11-09 01:17:13 [startup.py:38] 
(APIServer pid=8554) INFO 11-09 01:17:13 [startup.py:38]        █     █     █▄   ▄█
(APIServer pid=8554) INFO 11-09 01:17:13 [startup.py:38]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.33.0
(APIServer pid=8554) INFO 11-09 01:17:13 [startup.py:38]   █▄█▀ █     █     █     █  model   BSC-LT/salamandra-7b-instruct-2606
(APIServer pid=8554) INFO 11-09 01:17:13 [startup.py:38]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=8554) INFO 11-09 01:17:13 [startup.py:38] 
(APIServer pid=8554) INFO 11-09 01:17:13 [startup.py:96] non-default args: {'model_tag': 'BSC-LT/salamandra-7b-in

In [ ]:
import subprocess

#The previous cell was not addapted to collab
with open("vllm_server.log", "w") as archivo:
    servidor = subprocess.Popen(
        [
            "./vllm/bin/vllm",
            "serve",
            "BSC-LT/salamandra-7b-instruct-2606",
            "--max-model-len", "8192",
            "--port", "8000"
        ],
        stdout=archivo,
        stderr=subprocess.STDOUT,
        start_new_session=True
    )

Server iniciado: 13848


In [34]:
!curl http://127.0.0.1:8000/v1/models

{"object":"list","data":[{"id":"BSC-LT/salamandra-7b-instruct-2606","object":"model","created":1794215037,"owned_by":"vllm","root":"BSC-LT/salamandra-7b-instruct-2606","parent":null,"max_model_len":8192,"permission":[{"id":"modelperm-bca6abd3b3f464d1","object":"model_permission","created":1794215037,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}

In [38]:
!./speculators/bin/speculators regenerate-responses --dataset ./alia_prompts_pilot.jsonl --endpoint http://127.0.0.1:8000/v1/chat/completions --limit 4 --concurrency 1 --max-tokens 4096 --outfile ./alia_salamandra_test.jsonl

Using endpoint: http://127.0.0.1:8000/v1/chat/completions
Auto-detected model: BSC-LT/salamandra-7b-instruct-2606
Using model: BSC-LT/salamandra-7b-instruct-2606
Loading tokenizer: BSC-LT/salamandra-7b-instruct-2606
Using dataset: alia_prompts_pilot.jsonl
Split: train
Prompt field: prompt
Output file: ./alia_salamandra_test.jsonl
Error file: alia_salamandra_test.errors.jsonl


Pipeline complete in 35.6s: 4 conversations (4 ok, 0 errors, 0 truncated)
Throughput: 0.2 req/s, 86 tok/s | Avg request latency: 4451 ms



In [44]:
import json

with open("alia_salamandra_test.jsonl", encoding="utf-8") as archivo:
    resultados = [json.loads(linea) for linea in archivo]


In [45]:
print(resultados[-2]["text"])

<s><|im_start|>user
13. (ROM) Sea $P$ un polinomio de grado $n$ que satisface
$$
P(k)=\binom{n+1}{k}^{-1} \quad \text { para } k=0,1, \ldots, n
$$

Determina $P(n+1)$.<|im_end|>
<|im_start|>assistant
13. Demostramos por inducción matemática que $P(k)=\binom{n+1}{k}$ para $0 \leq k \leq n$. Esto es trivialmente cierto para $n=0$. Supongamos que es cierto para algún $n$, y consideremos el polinomio $P(x)-\binom{n+1}{x}$ de grado $n$ (que puede tener raíces repetidas). En la expansión por partes de su valor en $k=n+1$, obtenemos
$$
P(n+1)=\binom{n+2}{n+1}-\binom{n+1}{n} .
$$<|im_end|>


In [46]:
#Test con 100 respuesas generadas
!./speculators/bin/speculators regenerate-responses --dataset ./alia_prompts_pilot.jsonl --endpoint http://127.0.0.1:8000/v1/chat/completions --limit 100 --concurrency 4 --max-tokens 4096 --outfile ./alia_salamandra_100.jsonl

Using endpoint: http://127.0.0.1:8000/v1/chat/completions
Auto-detected model: BSC-LT/salamandra-7b-instruct-2606
Using model: BSC-LT/salamandra-7b-instruct-2606
Loading tokenizer: BSC-LT/salamandra-7b-instruct-2606
Using dataset: alia_prompts_pilot.jsonl
Split: train
Prompt field: prompt
Output file: ./alia_salamandra_100.jsonl
Error file: alia_salamandra_100.errors.jsonl


Pipeline complete in 183.7s: 100 conversations (100 ok, 0 errors, 2 truncated)
Throughput: 0.8 req/s, 314 tok/s | Avg request latency: 4882 ms



In [49]:
with open("alia_salamandra_100.jsonl", encoding="utf-8") as archivo:
    resultados = [json.loads(linea) for linea in archivo]

print("Respuestas:", len(resultados))
print("Acabadas:", Counter(
    r["metadata"]["finish_reason"] for r in resultados
))

Respuestas: 141
Acabadas: Counter({'stop': 141})


In [ ]:
!./speculators/bin/speculators prepare-data --model BSC-LT/salamandra-7b-instruct-2606 --data ./alia_salamandra_100.jsonl --output ./alia_eagle3_prepared --seq-length 8192